In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.data.preprocessing import load_daily

# Carichiamo i dati giornalieri definendo la variabile 'data' che mancava
data = load_daily()

# 1. Sort data chronologically
data = data.sort_index()

# 2. Define features and target
features = [
    "t2m_mean",
    "t2m_min",
    "t2m_max",
    "tp_mm",
    "e_mm",
    "pev_mm",
    "skt_mean",
    "stl1_mean",
    "stl2_mean",
    "stl3_mean",
    "stl4_mean",
    "swvl1_mean",
    "swvl2_mean",
    "swvl3_mean",
    "swvl4_mean"
]

# Use today's variables to predict the next day's mean temperature
X = data[features].copy()
y = data["t2m_mean"].shift(-1)

# Remove the last row because its next-day target is unavailable
X = X.iloc[:-1]
y = y.iloc[:-1]

# 3. Chronological split: 70% train, 15% validation, 15% test
n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

# 4. Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Validation evaluation
y_val_pred = model.predict(X_val)

print("\nValidation metrics:")
print("MAE:", mean_absolute_error(y_val, y_val_pred))
print("RMSE:", mean_squared_error(y_val, y_val_pred) ** 0.5)
print("R²:", r2_score(y_val, y_val_pred))

# 6. Final test evaluation
y_test_pred = model.predict(X_test)

print("\nTest metrics:")
print("MAE:", mean_absolute_error(y_test, y_test_pred))
print("RMSE:", mean_squared_error(y_test, y_test_pred) ** 0.5)
print("R²:", r2_score(y_test, y_test_pred))

# 7. Compare actual and predicted temperatures
results = pd.DataFrame({
    "Actual temperature (°C)": y_test,
    "Predicted temperature (°C)": y_test_pred
}, index=y_test.index)

display(results.head(10))